## Fase 1: Ingesta y Limpieza de Datos SQL (Archivos CSV)
En esta sección, vamos a cargar los datos demográficos y de envejecimiento. El objetivo de la limpieza aquí será:

Identificar si existen valores nulos.
2. Estandarizar los nombres de los países (quitar espacios en blanco, poner todo en mayúsculas o minúsculas) para que la futura integración sea exitosa.

In [4]:
import pandas as pd
import json
import numpy as np
import warnings

# Silenciar FutureWarnings de pandas
warnings.filterwarnings('ignore', category=FutureWarning)

# 1. Cargar los datos desde la carpeta de SQL
ruta_sql = 'data/Datos_para_SQL/'

df_poblacion = pd.read_csv(ruta_sql + 'pais_poblacion.csv')
df_envejecimiento = pd.read_csv(ruta_sql + 'pais_envejecimiento.csv')

# 2. Exploración inicial (Historial de limpieza)
print("=" * 60)
print("ANALISIS INICIAL DE DATOS CSV")
print("=" * 60)
print("\n--- Nulos en Poblacion ---")
print(df_poblacion.isnull().sum())
print(f"\nTotal de registros: {len(df_poblacion)}")

print("\n--- Nulos en Envejecimiento ---")
print(df_envejecimiento.isnull().sum())
print(f"\nTotal de registros: {len(df_envejecimiento)}")

# 3. LIMPIEZA AVANZADA - DataFrame de Población
print("\n" + "=" * 60)
print("LIMPIEZA: DataFrame de Poblacion")
print("=" * 60)

# Estandarizar nombres de países
columna_pais_poblacion = 'pais'
df_poblacion[columna_pais_poblacion] = df_poblacion[columna_pais_poblacion].astype(str).str.strip().str.lower()

# Rellenar valores numéricos nulos con la mediana de cada columna
columnas_numericas_pob = ['poblacion', 'costo_bajo_hospedaje', 'costo_promedio_comida', 
                          'costo_bajo_transporte', 'costo_promedio_entretenimiento']

for col in columnas_numericas_pob:
    if df_poblacion[col].isnull().sum() > 0:
        mediana = df_poblacion[col].median()
        df_poblacion[col].fillna(mediana, inplace=True)
        print(f"- Rellenados {df_poblacion[col].isnull().sum()} nulos en '{col}' con mediana: {mediana:.2f}")

# Rellenar valores categóricos nulos con el valor más frecuente
if df_poblacion['continente'].isnull().sum() > 0:
    moda_continente = df_poblacion['continente'].mode()[0]
    df_poblacion['continente'].fillna(moda_continente, inplace=True)
    print(f"- Rellenados nulos en 'continente' con: {moda_continente}")

# 4. LIMPIEZA AVANZADA - DataFrame de Envejecimiento
print("\n" + "=" * 60)
print("LIMPIEZA: DataFrame de Envejecimiento")
print("=" * 60)

# Estandarizar nombres de países
columna_pais_envejecimiento = 'nombre_pais'
df_envejecimiento[columna_pais_envejecimiento] = df_envejecimiento[columna_pais_envejecimiento].astype(str).str.strip().str.lower()

# Rellenar población con la mediana
if df_envejecimiento['poblacion'].isnull().sum() > 0:
    mediana_pob = df_envejecimiento['poblacion'].median()
    nulos_antes = df_envejecimiento['poblacion'].isnull().sum()
    df_envejecimiento['poblacion'].fillna(mediana_pob, inplace=True)
    print(f"- Rellenados {nulos_antes} nulos en 'poblacion' con mediana: {mediana_pob:.0f}")

# Rellenar capital con "Desconocida"
if df_envejecimiento['capital'].isnull().sum() > 0:
    nulos_antes = df_envejecimiento['capital'].isnull().sum()
    df_envejecimiento['capital'].fillna('Desconocida', inplace=True)
    print(f"- Rellenados {nulos_antes} nulos en 'capital' con: 'Desconocida'")

# Rellenar continente y región usando df_poblacion como referencia
# Crear un diccionario de mapeo de país -> continente desde df_poblacion
if df_envejecimiento['continente'].isnull().sum() > 0:
    mapa_continente = df_poblacion.set_index('pais')['continente'].to_dict()
    
    # Intentar rellenar desde el mapeo
    def obtener_continente(row):
        if pd.isna(row['continente']) and row['nombre_pais'] in mapa_continente:
            return mapa_continente[row['nombre_pais']]
        elif pd.isna(row['continente']):
            return 'Desconocido'
        return row['continente']
    
    df_envejecimiento['continente'] = df_envejecimiento.apply(obtener_continente, axis=1)
    print(f"- Rellenados nulos en 'continente' usando referencia cruzada y 'Desconocido'")

# Rellenar región con "Desconocida"
if df_envejecimiento['region'].isnull().sum() > 0:
    nulos_antes = df_envejecimiento['region'].isnull().sum()
    df_envejecimiento['region'].fillna('Desconocida', inplace=True)
    print(f"- Rellenados {nulos_antes} nulos en 'region' con: 'Desconocida'")

# 5. Verificación final
print("\n" + "=" * 60)
print("VERIFICACION POST-LIMPIEZA")
print("=" * 60)
print("\n--- Nulos restantes en Poblacion ---")
print(df_poblacion.isnull().sum().sum(), "nulos en total")

print("\n--- Nulos restantes en Envejecimiento ---")
print(df_envejecimiento.isnull().sum().sum(), "nulos en total")

print("\nLimpieza de SQL completada exitosamente!")

# 6. Guardar los DataFrames limpios con codificacion UTF-8-BOM para compatibilidad con Excel
print("\n" + "=" * 60)
print("GUARDANDO ARCHIVOS LIMPIOS")
print("=" * 60)

# Usar encoding='utf-8-sig' para que Excel reconozca correctamente los caracteres especiales
df_poblacion.to_csv(ruta_sql + 'pais_poblacion_limpio.csv', index=False, encoding='utf-8-sig')
print(f"- Guardado: {ruta_sql}pais_poblacion_limpio.csv (codificacion: UTF-8 con BOM)")

df_envejecimiento.to_csv(ruta_sql + 'pais_envejecimiento_limpio.csv', index=False, encoding='utf-8-sig')
print(f"- Guardado: {ruta_sql}pais_envejecimiento_limpio.csv (codificacion: UTF-8 con BOM)")

print("\nNOTA: Archivos guardados con UTF-8-BOM para compatibilidad con Excel")

print("\n--- Muestra DataFrame de Poblacion ---")
display(df_poblacion.head(3))
print("\n--- Muestra DataFrame de Envejecimiento ---")
display(df_envejecimiento.head(3))

ANALISIS INICIAL DE DATOS CSV

--- Nulos en Poblacion ---
_id                               0
continente                        0
pais                              0
poblacion                         0
costo_bajo_hospedaje              0
costo_promedio_comida             0
costo_bajo_transporte             0
costo_promedio_entretenimiento    0
dtype: int64

Total de registros: 106

--- Nulos en Envejecimiento ---
id_pais                     0
nombre_pais                 0
capital                   101
continente                101
region                    101
poblacion                 101
tasa_de_envejecimiento      0
dtype: int64

Total de registros: 106

LIMPIEZA: DataFrame de Poblacion

LIMPIEZA: DataFrame de Envejecimiento
- Rellenados 101 nulos en 'poblacion' con mediana: 5450000
- Rellenados 101 nulos en 'capital' con: 'Desconocida'
- Rellenados nulos en 'continente' usando referencia cruzada y 'Desconocido'
- Rellenados 101 nulos en 'region' con: 'Desconocida'

VERIFICACION POS

,_id,continente,pais,poblacion,costo_bajo_hospedaje,costo_promedio_comida,costo_bajo_transporte,costo_promedio_entretenimiento
0,67e2130bc29aa80afae47e7f,África,ruanda,12952218,7.0,17.0,17.0,45.0
1,67e2130bc29aa80afae47e80,África,namibia,2540905,8.0,25.0,11.0,14.0
2,67e2130bc29aa80afae47e7a,África,mozambique,31255435,9.0,41.0,9.0,32.0



--- Muestra DataFrame de Envejecimiento ---


,id_pais,nombre_pais,capital,continente,region,poblacion,tasa_de_envejecimiento
0,1,latvia,Riga,Europa,Norte de Europa,1906800.0,12.49
1,2,romania,Bucarest,Europa,Europa del Este,19000000.0,24.01
2,3,kosovo,Pristina,Europa,Balcanes,1800000.0,19.64


## Fase 2: Ingesta y Limpieza de Datos NoSQL (Archivos JSON)
Aquí procesaremos los datos de costos turísticos y el índice Big Mac. Las tareas principales son:

Unificar los 4 archivos de continentes en un solo DataFrame.

Estandarizar los nombres de los países con la misma regla usada en la Fase 1.

Revisar tipos de datos (asegurarnos de que los costos sean valores numéricos y no texto).

In [5]:
# 1. Cargar los datos desde la carpeta de Mongo
ruta_mongo = 'data/Datos_para_MongoDB/'

archivos_turismo = [
   'costos_turisticos_africa.json',
   'costos_turisticos_america.json',
   'costos_turisticos_asia.json',
   'costos_turisticos_europa.json'
]

print("=" * 60)
print("CARGA Y LIMPIEZA DE DATOS JSON")
print("=" * 60)

# Leer y concatenar los 4 archivos de turismo
lista_df_turismo = []
for archivo in archivos_turismo:
   with open(ruta_mongo + archivo, 'r', encoding='utf-8') as f:
      datos = json.load(f)
      lista_df_turismo.append(pd.DataFrame(datos))
      print(f"- Cargado: {archivo} ({len(datos)} registros)")

df_turismo = pd.concat(lista_df_turismo, ignore_index=True)
print(f"\n- Total de registros de turismo: {len(df_turismo)}")

# Leer el índice Big Mac
with open(ruta_mongo + 'paises_mundo_big_mac.json', 'r', encoding='utf-8') as f:
   datos_big_mac = json.load(f)
   df_big_mac = pd.DataFrame(datos_big_mac)
   print(f"- Cargado: paises_mundo_big_mac.json ({len(df_big_mac)} registros)")

# 2. APLANAR LA ESTRUCTURA ANIDADA DE COSTOS TURISTICOS
print("\n" + "=" * 60)
print("APLANAMIENTO DE ESTRUCTURA ANIDADA")
print("=" * 60)

# Extraer los costos del diccionario anidado
def aplanar_costos(row):
    """Extrae los valores de costos de la estructura anidada"""
    costos = row['costos_diarios_estimados_en_dólares']
    
    return pd.Series({
        'hospedaje_bajo': costos['hospedaje']['precio_bajo_usd'],
        'hospedaje_promedio': costos['hospedaje']['precio_promedio_usd'],
        'hospedaje_alto': costos['hospedaje']['precio_alto_usd'],
        'comida_bajo': costos['comida']['precio_bajo_usd'],
        'comida_promedio': costos['comida']['precio_promedio_usd'],
        'comida_alto': costos['comida']['precio_alto_usd'],
        'transporte_bajo': costos['transporte']['precio_bajo_usd'],
        'transporte_promedio': costos['transporte']['precio_promedio_usd'],
        'transporte_alto': costos['transporte']['precio_alto_usd'],
        'entretenimiento_bajo': costos['entretenimiento']['precio_bajo_usd'],
        'entretenimiento_promedio': costos['entretenimiento']['precio_promedio_usd'],
        'entretenimiento_alto': costos['entretenimiento']['precio_alto_usd']
    })

# Aplicar la función para aplanar
df_costos_aplanados = df_turismo.apply(aplanar_costos, axis=1)

# Combinar con las columnas base (sin la columna de costos anidada)
df_turismo_limpio = pd.concat([
    df_turismo[['continente', 'región', 'país', 'capital', 'población']], 
    df_costos_aplanados
], axis=1)

print("- Estructura aplanada exitosamente")
print(f"  Columnas antes: {df_turismo.shape[1]} -> Columnas despues: {df_turismo_limpio.shape[1]}")

# 3. LIMPIEZA Y ESTANDARIZACION
print("\n" + "=" * 60)
print("ESTANDARIZACION Y LIMPIEZA")
print("=" * 60)

# Los archivos JSON usan 'país' con tilde
columna_pais_json = 'país'

# Estandarizar nombres: quitar espacios a los lados y convertir a minúsculas
df_turismo_limpio[columna_pais_json] = df_turismo_limpio[columna_pais_json].astype(str).str.strip().str.lower()
df_big_mac[columna_pais_json] = df_big_mac[columna_pais_json].astype(str).str.strip().str.lower()

print("- Nombres de paises estandarizados (minusculas, sin espacios)")

# 4. MANEJO DE VALORES NULOS
print("\n--- Analisis de Nulos en Turismo ---")
nulos_turismo = df_turismo_limpio.isnull().sum()
print(nulos_turismo[nulos_turismo > 0] if nulos_turismo.sum() > 0 else "- No hay valores nulos")

print("\n--- Analisis de Nulos en Big Mac ---")
nulos_bigmac = df_big_mac.isnull().sum()
print(nulos_bigmac[nulos_bigmac > 0] if nulos_bigmac.sum() > 0 else "- No hay valores nulos")

# Rellenar valores numéricos nulos en turismo (si existen) con la mediana
columnas_numericas_turismo = [col for col in df_turismo_limpio.columns 
                               if col not in ['continente', 'región', 'país', 'capital']]

for col in columnas_numericas_turismo:
    if df_turismo_limpio[col].isnull().sum() > 0:
        mediana = df_turismo_limpio[col].median()
        nulos_antes = df_turismo_limpio[col].isnull().sum()
        df_turismo_limpio[col].fillna(mediana, inplace=True)
        print(f"- Rellenados {nulos_antes} nulos en '{col}' con mediana: {mediana:.2f}")

# Rellenar valores categóricos nulos (si existen)
for col in ['continente', 'región', 'capital']:
    if df_turismo_limpio[col].isnull().sum() > 0:
        nulos_antes = df_turismo_limpio[col].isnull().sum()
        df_turismo_limpio[col].fillna('Desconocido', inplace=True)
        print(f"- Rellenados {nulos_antes} nulos en '{col}' con: 'Desconocido'")

# Rellenar nulos en Big Mac
if df_big_mac['precio_big_mac_usd'].isnull().sum() > 0:
    mediana_bigmac = df_big_mac['precio_big_mac_usd'].median()
    nulos_antes = df_big_mac['precio_big_mac_usd'].isnull().sum()
    df_big_mac['precio_big_mac_usd'].fillna(mediana_bigmac, inplace=True)
    print(f"- Rellenados {nulos_antes} nulos en precio Big Mac con mediana: ${mediana_bigmac:.2f}")

# 5. ESTADISTICAS FINALES
print("\n" + "=" * 60)
print("RESUMEN FINAL")
print("=" * 60)
print(f"\nDataFrame Turismo:")
print(f"   - Registros: {len(df_turismo_limpio)}")
print(f"   - Columnas: {len(df_turismo_limpio.columns)}")
print(f"   - Nulos totales: {df_turismo_limpio.isnull().sum().sum()}")

print(f"\nDataFrame Big Mac:")
print(f"   - Registros: {len(df_big_mac)}")
print(f"   - Columnas: {len(df_big_mac.columns)}")
print(f"   - Nulos totales: {df_big_mac.isnull().sum().sum()}")

# 6. Guardar los DataFrames limpios
print("\n" + "=" * 60)
print("GUARDANDO ARCHIVOS LIMPIOS")
print("=" * 60)

# Convertir DataFrame a formato de lista de diccionarios para JSON
turismo_json = df_turismo_limpio.to_dict('records')
with open(ruta_mongo + 'costos_turisticos_limpio.json', 'w', encoding='utf-8') as f:
    json.dump(turismo_json, f, ensure_ascii=False, indent=2)
print(f"- Guardado: {ruta_mongo}costos_turisticos_limpio.json")

big_mac_json = df_big_mac.to_dict('records')
with open(ruta_mongo + 'paises_mundo_big_mac_limpio.json', 'w', encoding='utf-8') as f:
    json.dump(big_mac_json, f, ensure_ascii=False, indent=2)
print(f"- Guardado: {ruta_mongo}paises_mundo_big_mac_limpio.json")

print("\n--- Muestra DataFrame de Turismo (aplanado) ---")
display(df_turismo_limpio.head(3))

print("\n--- Muestra DataFrame de Big Mac ---")
display(df_big_mac.head(3))

# Guardar el DataFrame aplanado para uso futuro
df_turismo = df_turismo_limpio

CARGA Y LIMPIEZA DE DATOS JSON
- Cargado: costos_turisticos_africa.json (20 registros)
- Cargado: costos_turisticos_america.json (20 registros)
- Cargado: costos_turisticos_asia.json (20 registros)
- Cargado: costos_turisticos_europa.json (46 registros)

- Total de registros de turismo: 106
- Cargado: paises_mundo_big_mac.json (106 registros)

APLANAMIENTO DE ESTRUCTURA ANIDADA
- Estructura aplanada exitosamente
  Columnas antes: 6 -> Columnas despues: 17

ESTANDARIZACION Y LIMPIEZA
- Nombres de paises estandarizados (minusculas, sin espacios)

--- Analisis de Nulos en Turismo ---
- No hay valores nulos

--- Analisis de Nulos en Big Mac ---
- No hay valores nulos

RESUMEN FINAL

DataFrame Turismo:
   - Registros: 106
   - Columnas: 17
   - Nulos totales: 0

DataFrame Big Mac:
   - Registros: 106
   - Columnas: 3
   - Nulos totales: 0

GUARDANDO ARCHIVOS LIMPIOS
- Guardado: data/Datos_para_MongoDB/costos_turisticos_limpio.json
- Guardado: data/Datos_para_MongoDB/paises_mundo_big_mac_lim

,continente,región,país,capital,población,hospedaje_bajo,hospedaje_promedio,hospedaje_alto,comida_bajo,comida_promedio,comida_alto,transporte_bajo,transporte_promedio,transporte_alto,entretenimiento_bajo,entretenimiento_promedio,entretenimiento_alto
0,África,África Austral,sudáfrica,Pretoria,59308690,21.0,30.0,39.0,14.0,20.0,26.0,25.0,37.0,48.0,30.0,43.0,55.0
1,África,África Occidental,nigeria,Abuja,206139589,28.0,41.0,53.0,15.0,22.0,28.0,15.0,22.0,28.0,19.0,28.0,36.0
2,África,África del Norte,egipto,El Cairo,102334404,26.0,38.0,49.0,7.0,10.0,13.0,29.0,42.0,54.0,23.0,33.0,42.0



--- Muestra DataFrame de Big Mac ---


,país,continente,precio_big_mac_usd
0,albania,Europa,5.79
1,andorra,Europa,2.08
2,austria,Europa,5.55
